# Modulo 1 — Data Engine

Notebook **Kaggle-compatibile** che dimostra la pipeline di importazione, pulizia, resampling multi-timeframe e normalizzazione.

La logica vive nel package riutilizzabile `trading_ai` (cartella `trading_ai/` del repo). Qui lo orchestriamo soltanto, così il notebook resta sottile e i test automatici coprono il codice vero.

**Pipeline:** import → pulizia automatica → resample (M1→H1/H4/D1) → normalizzazione anti-leakage → salvataggio Parquet.

## 0. Setup

Su Kaggle: aggiungi il repo come *Utility Script* o clona via Git, poi assicurati che la cartella del repo sia nel `sys.path`. In locale basta eseguire dalla root del progetto.

In [ ]:
import sys, os
from pathlib import Path

# Rendiamo importabile il package `trading_ai` sia in locale sia su Kaggle.
# Cerchiamo la root del repo risalendo dalle posizioni tipiche.
for candidate in [Path.cwd(), Path.cwd().parent, Path('/kaggle/working/trading-ai')]:
    if (candidate / 'trading_ai').exists():
        sys.path.insert(0, str(candidate))
        break

import trading_ai
print('trading_ai version:', trading_ai.__version__)

## 1. Import dei dati

In produzione carichi un CSV reale (es. esportazione MetaTrader):

```python
df = eng.load_csv('datasets/EURUSD_M1.csv', has_header=True)
```

Qui usiamo dati **sintetici** così il notebook gira ovunque senza download esterni.

In [ ]:
from trading_ai.data_engine import DataEngine, generate_ohlcv

# Generiamo ~200k candele M1 (qualche mese di mercato) per mostrare la scalabilità.
raw = generate_ohlcv(n=200_000, start='2022-01-01', freq='1min', seed=7)
print('Candele grezze:', len(raw))
raw.head()

## 2. Pulizia automatica

Iniettiamo qualche riga corrotta per vedere il cleaner all'opera. In dati reali questi problemi (NaN, prezzi impossibili, OHLC incoerenti, duplicati) capitano di continuo.

In [ ]:
import numpy as np

dirty = raw.copy()
dirty.iloc[100, dirty.columns.get_loc('high')] = dirty.iloc[100]['low'] - 1  # high<low
dirty.iloc[200, dirty.columns.get_loc('close')] = -1.0                        # prezzo negativo
dirty.iloc[300, dirty.columns.get_loc('open')] = np.nan                       # NaN

eng = DataEngine(max_return=0.20)
clean_df = eng.load_dataframe(dirty)   # normalizza schema + pulisce
print(eng.last_report.as_dict())
clean_df.head()

## 3. Resampling multi-timeframe

Dal timeframe base M1 generiamo H1, H4 e D1 in un colpo solo. Le regole di aggregazione sono quelle standard (open=first, high=max, low=min, close=last, volume=sum).

In [ ]:
frames = eng.to_timeframes(clean_df, ['H1', 'H4', 'D1'])
for tf, f in frames.items():
    print(f'{tf}: {len(f)} barre')
frames['H1'].head()

## 4. Normalizzazione anti-leakage

Split temporale train/test, `fit` solo sul training, `transform` su entrambi. Per il ML usiamo i **rendimenti log** (`returns`), che rendono la serie stazionaria.

In [ ]:
from trading_ai.data_engine import Normalizer

h1 = frames['H1']
split = int(len(h1) * 0.7)
train, test = h1.iloc[:split], h1.iloc[split:]

norm = Normalizer(method='zscore', columns=['close']).fit(train)  # impara SOLO dal train
train_n = norm.transform(train)
test_n = norm.transform(test)   # usa i parametri del train -> nessun leakage
print('media close normalizzata (train):', round(float(train_n['close'].mean()), 4))
test_n[['close']].head()

## 5. Salvataggio dataset

Salviamo in **Parquet** (compresso, tipizzato, veloce). Su Kaggle scrivi in `/kaggle/working`.

In [ ]:
out_dir = Path(os.environ.get('TRADING_AI_ROOT', '.')) / 'datasets'
out_dir.mkdir(parents=True, exist_ok=True)
path = eng.save(frames['H1'], out_dir / 'SYNTH_H1.parquet')
print('Salvato:', path)

# Verifica round-trip: ricarico e confronto le dimensioni.
reloaded = eng.load(path)
print('Ricaricato:', reloaded.shape)

## ✅ Modulo 1 verificato

Import → pulizia → resample → normalizzazione → persistenza funzionano. I test automatici (`pytest tests/test_data_engine.py`) coprono ogni passaggio.

**Prossimo:** Modulo 2 — Feature Engineering (ATR, RSI, MACD, ADX, EMA/SMA, VWAP, Bollinger, pattern, FVG, BOS/CHoCH, swing, S/R, liquidità, volatilità, trend strength).